# 02 – Validation Setup

This notebook designs the **validation scheme** used to choose every model in the project: **expanding-window, time-based folds** that train on the past and validate on the future, plus four **late folds** that stop a one-off illicit pattern at step 24 from decidiing model comparisons.

## Goal
Before comparing models, we need a fair estimate of **how well a model will score on the hidden test set**.

The EDA (`01_eda.ipynb`) showed two things that shape this choice:
- The test set comes from **later** time steps (36–49) than the training data (1–35).
- The share of illicit transactions **changes a lot over time** (0.4% to 36% per time step).

The validation must therefore mimic the real task: **train on the past, predict the future**.

## What this notebook does
1. Defines time-based folds (train on earlier time steps, validate on later ones).
2. Shows why the common alternative, a **random** split, gives over-optimistic scores here.
3. Finds that the first fold is decided by a single time step, and adds four **late folds** that avoid this.

Comparing models is left to `03_baseline_models.ipynb`. Here, two simple models serve only as measuring tools.

## 0. Setup
The validation logic lives in `src/validation.py` and the two measuring-tool models in `src/models.py`, so every notebook evaluates models in exactly the same way.

**Helpers**
- `time_splits(df, folds)`: yields (fold number, training rows, validation rows) for each fold.
- `evaluate(make_model, df, features, folds)`: trains a fresh model on each fold and returns one row of results (AUC, sizes) per fold.
- `FOLDS` / `LATE_FOLDS`: the three early folds and the four late folds, as (last training step, (first, last validation step)).
- `make_logreg`, `make_random_forest`: build a new, untrained model each time they are called, so no fold reuses a fitted model.

In [1]:
import sys
sys.path.append("..")  # add the repo root so `src` can be imported from inside notebooks/

import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.data import load_train, labeled, FEATURE_COLS, LABEL_COL, TIME_COL, SEED
from src.models import make_logreg, make_random_forest
from src.validation import LATE_FOLDS, time_splits, evaluate

## 1. Load the labeled training data
Only labeled transactions can be used to fit and score a supervised model.

In [2]:
lab = labeled(load_train())  # rows with a known label: 1 = illicit, 0 = licit
print(f"Labeled transactions: {len(lab):,} (time steps {lab[TIME_COL].min()}-{lab[TIME_COL].max()})")

Labeled transactions: 31,235 (time steps 1-35)


## 2. Time-based folds
I use **expanding-window** validation: each fold trains on all time steps up to a cut-off and validates on the next five steps.
The training window grows from fold to fold, just like the final model, which trains on steps 1–35 and predicts steps 36–49.

| Fold | Train on | Validate on |
|---|---|---|
| 1 | steps 1–20 | steps 21–25 |
| 2 | steps 1–25 | steps 26–30 |
| 3 | steps 1–30 | steps 31–35 |

Using three folds instead of one shows how **stable** a model is across different future periods, not just how well it does on one favourable period.

**Example:** fold 2 fits on the 24,200 labeled rows of steps 1-25 and computes one AUC over the 2,705 labeled rows of steps 26-30 (sizes printed below).

In [3]:
# Size of each fold: a validation set needs enough illicit cases for a reliable AUC
fold_sizes = pd.DataFrame([
    {"fold": i,
     "train_rows": len(tr), "train_illicit": int(tr[LABEL_COL].sum()),
     "valid_rows": len(va), "valid_illicit": int(va[LABEL_COL].sum())}
    for i, tr, va in time_splits(lab)
]).set_index("fold")
fold_sizes

,train_rows,train_illicit,valid_rows,valid_illicit
fold,,,,
1,18889,1771,5311,566
2,24200,2337,2705,617
3,26905,2954,4330,690


**Takeaway:** Every validation period contains several hundred illicit transactions (566 to 690), enough for a stable AUC estimate.

## 3. Why not a random split?
A random split (for example 5-fold cross-validation with shuffled rows) mixes past and future transactions,
so the model is partly tested on time periods it has already seen in training. Under drift, this **overestimates** performance.

A validation scheme can only be tested with a model, so I use two simple models from `src/models.py` as measuring tools:
- **Logistic regression**: a linear model on standardised features.
- **Random forest**: an ensemble of decision trees, which needs no scaling.

In [4]:
# Functions that build a fresh, untrained model each time they are called (defined in src/models.py)
models = {"Logistic regression": make_logreg, "Random forest": make_random_forest}

In [ ]:
comparison, fold_results = [], {}
for name, make_model in models.items():
    # Random split: 5 shuffled folds, each keeping the same illicit share (stratified)
    random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    random_auc = cross_val_score(make_model(), lab[FEATURE_COLS], lab[LABEL_COL],
                                 cv=random_cv, scoring="roc_auc").mean()

    # Time-based split: the three folds defined in section 2
    fold_results[name] = evaluate(make_model, lab, FEATURE_COLS)
    comparison.append({"model": name,
                       "random split AUC": round(random_auc, 4),
                       "time-based AUC (mean of 3 folds)": round(fold_results[name]["auc"].mean(), 4)})

comparison = pd.DataFrame(comparison).set_index("model")
comparison

,random split AUC,time-based AUC (mean of 3 folds)
model,,
Logistic regression,0.9760,0.9193
Random forest,0.9977,0.9529


**Takeaway:** The random split gives clearly higher AUCs than the time-based folds for both models (random forest: 0.9977 vs 0.9529).
A random split would overstate performance and could favour models that do not hold up on future data, so all later notebooks use time-based folds.

### Results per fold
Each fold on its own shows how much performance varies between future periods.

In [6]:
per_fold = pd.concat(fold_results, names=["model"]).reset_index(level=0)
per_fold.pivot(index="fold", columns="model", values="auc").round(4)

model,Logistic regression,Random forest
fold,,
1,0.8975,0.8691
2,0.9316,0.9926
3,0.9288,0.9970


**Takeaway:** Scores differ a lot between folds. The random forest is near-perfect on the later folds (0.9926 and 0.997) but drops to 0.8691 on fold 1 (validating on steps 21–25): A model trained on one period can fail on the next when illicit behaviour changes.
The next section looks at what causes the drop.

## 4. What decides fold 1?
Fold 1 validates on five time steps at once. I score each of them separately with the random forest trained on steps 1–20.
If one step scores far below the others, that step alone decides the fold.

In [7]:
# Train on fold 1's training steps, then score every validation row
fold1_train = lab[lab[TIME_COL] <= 20]
fold1_valid = lab[lab[TIME_COL].between(21, 25)].copy()
forest = make_random_forest().fit(fold1_train[FEATURE_COLS], fold1_train[LABEL_COL])
fold1_valid["score"] = forest.predict_proba(fold1_valid[FEATURE_COLS])[:, 1]  # probability of illicit

# AUC of each validation step on its own
per_step = pd.DataFrame([
    {"time_step": step, "illicit": int(rows[LABEL_COL].sum()), "AUC": round(roc_auc_score(rows[LABEL_COL], rows["score"]), 4)}
    for step, rows in fold1_valid.groupby(TIME_COL)
]).set_index("time_step")
print(per_step)

without_24 = fold1_valid[fold1_valid[TIME_COL] != 24]
print(f"\nFold 1 AUC, all steps:       {roc_auc_score(fold1_valid[LABEL_COL], fold1_valid['score']):.4f}")
print(f"Fold 1 AUC, without step 24: {roc_auc_score(without_24[LABEL_COL], without_24['score']):.4f}")

           illicit     AUC
time_step                 
21             100  0.9924
22             158  0.9948
23              53  0.9034
24             137  0.4815
25             118  0.9995

Fold 1 AUC, all steps:       0.8691
Fold 1 AUC, without step 24: 0.9834


Is step 24 simply hard, or is it new? I train the same model on steps 1–23 instead, so it has seen the three steps just before step 24.

In [8]:
train_to_23 = lab[lab[TIME_COL] <= 23]
step_24 = lab[lab[TIME_COL] == 24]
forest_23 = make_random_forest().fit(train_to_23[FEATURE_COLS], train_to_23[LABEL_COL])
print(f"Step 24 AUC, trained on steps 1-23: {roc_auc_score(step_24[LABEL_COL], forest_23.predict_proba(step_24[FEATURE_COLS])[:, 1]):.4f}")

Step 24 AUC, trained on steps 1-23: 0.9856


**Takeaway:** Trained on steps 1–20, the random forest ranks step 24's 137 illicit transactions no better than chance (AUC 0.48), while every other step scores 0.90 or more. Without step 24, fold 1 scores 0.9834 instead of 0.8691.
Once steps 21–23 are in training, step 24 scores 0.9856. So step 24 is not hard in itself: it holds a **new kind of illicit transaction** that first appears after step 20.

Fold 1 therefore mostly measures one thing: whether a model happens to handle this one new pattern without having seen it. That is a matter of luck with respect to this particular pattern, and it does not tell us which model handles the test period best.
The test period may well contain new patterns of its own, but no validation fold can predict how a model reacts to a pattern it has never seen.

## 5. Late folds
To compare models, I add four **late folds**. Each trains on all steps up to a cut-off and validates on **all** remaining training steps up to 35:

| Fold | Train on | Validate on |
|---|---|---|
| 1 | steps 1–24 | steps 25–35 |
| 2 | steps 1–26 | steps 27–35 |
| 3 | steps 1–28 | steps 29–35 |
| 4 | steps 1–30 | steps 31–35 |

**Design choices**
1. **Step 24 is always in training.** Every training window contains steps 21–24, so no fold is decided by the step-24 event.
- **Long, pooled validation periods.** Each validation period spans 5 to 11 steps, like the 14-step test period, and the AUC is computed over all of its steps together, as on the leaderboard.

The folds overlap (all four validate on steps 31–35), so they are four views of the same late period rather than four independent tests.

In [9]:
late_sizes = pd.DataFrame([
    {"fold": i,
     "train_rows": len(tr), "train_illicit": int(tr[LABEL_COL].sum()),
     "valid_rows": len(va), "valid_illicit": int(va[LABEL_COL].sum())}
    for i, tr, va in time_splits(lab, LATE_FOLDS)
]).set_index("fold")
late_sizes

,train_rows,train_illicit,valid_rows,valid_illicit
fold,,,,
1,23606,2219,7629,1425
2,24717,2433,6518,1211
3,25207,2542,6028,1102
4,26905,2954,4330,690


**Takeaway:** Each late fold validates on 690 to 1,425 illicit transactions (early folds: 566 to 690), spread over 5 to 11 time steps.

Strong models score close to 1.0 on these folds, so differences between good models will be small (in the third or fourth decimal). I use the late folds to tell **which** model is better, not to predict its leaderboard score.

From `03_baseline_models.ipynb` onwards, models are compared by their **mean AUC on the late folds**, and every run is logged in `experiments.csv`. Notebook 05 checks how well this choice matched the public leaderboard.

## 6. Summary
| Decision | Reason |
|---|---|
| Never use random splits | They mix past and future and overestimate performance (random forest: 0.9977 random vs 0.9529 time-based) |
| Validate with time-based folds: train on earlier steps, validate on later ones | The test set lies in the future and the data drifts over time |
| Choose models by the **mean AUC on 4 late folds** (validate on 25–35, 27–35, 29–35, 31–35) | The first early fold is decided by step 24 alone (random forest 0.48 on step 24, 0.9834 on the rest of the fold) |
| Keep the 3 early folds as a second view | They show how a model reacts when the illicit pattern changes, but say little about which model is better |
| From notebook 03 onwards, log every model in `experiments.csv` | Keeps results comparable and ready for the report |

**Next:** `03_baseline_models.ipynb` compares standard classifiers on these folds.